# Session 3 — The LLM-based agent: retrieve, remember, plan, then act

**Salih Tutun, PhD** · DAT 5668 · Module 2 · Concept session (about 80 minutes)

I am Salih Tutun. Same Lumen Home desk as Monday. Maya's mug, Jordan's change of mind, Priya's order 1003. We are not leaving Agentic AI. Retrieval, memory, and a plan are how this **LLM-based agent** observes and decides. It still has to **act**.

Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Guess to a neighbor (20 seconds). Two voices share. Then type.

**Family this session: LLM-based agent.** Session 4 is the other family (RL), still the same loop. Do not mix them because both are "AI."

![An LLM-based agent is still the loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)

| What we practice | Loop box |
| --- | --- |
| Retrieve a policy chunk | Observe (a **tool**) |
| Hold the citation | Short-term memory |
| Retrieve → cite → act → release | Plan = decide |
| Refund, refuse, escalate | Action *a* |
| Was the citation real? Leakage? | Reward *r*, next ticket *s'* |

## Run of show
- 10 min — Recap Monday. Cold call: what did `lookup_order` observe that the chatbot skipped?
- 15 min — Why retrieval exists. Guess the SOC 2 question **before** we run it.
- 20 min — Near miss: which chunk comes first? Then the three-role handoff: who releases?
- 20 min — Your question, in your own words. Then one line for your project framework.
- 15 min — Walking-out answers, and why Session 4 is not this job.

We will not list the walking-out answers at the top. You will earn them.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I tell you what we are looking at. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor (20 seconds). Two people share. Then type `YOU TYPE HERE`. Then run. |
| **RUN WITH ME** | Already written. Run it. Do not edit. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** out loud — and one line you will keep for your project. |

There is no `TODO` in a concept session. Practice sessions (even numbers) are where you write the function.


## 1. Why retrieval exists — RUN WITH ME

A model that answers from memory will invent a refund window. Retrieval-augmented generation (RAG) forces the answer to come from **your** documents.

![Ask, cite, or miss](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-policy-chunks.png)

Same instructional policy as Monday's desk (not a real firm). P1 is Maya's cracked mug. P2 is Jordan's change of mind.

Run this cell. Do not edit it. Then **STOP**.


In [ ]:
# RUN WITH ME. Each string is one chunk. In a real stack you split long files
# on purpose (headings, or a fixed token length) and keep the source for the citation.
CHUNKS = [
    {"id": "P1", "source": "Refund policy", "text": "A refund for damaged items is allowed within 30 days of delivery. The amount is the order total."},
    {"id": "P2", "source": "Refund policy", "text": "A refund for a change of mind is not allowed. Offer store credit within 14 days instead."},
    {"id": "P3", "source": "Privacy", "text": "Do not paste customer addresses or payment numbers into a public chatbot."},
    {"id": "P4", "source": "Shipping", "text": "Lost shipments may be replaced after 7 business days. Refunds for lost shipments require a carrier trace."},
]


def tokens(text):
    cleaned = [word.strip(".,!?;:\"'").lower() for word in text.split()]
    return [word for word in cleaned if len(word) > 2]


def retrieve(query, k=2):
    """Score chunks by shared tokens. Simple on purpose so the failure mode is visible."""
    query_terms = set(tokens(query))
    scored = []
    for chunk in CHUNKS:
        overlap = query_terms & set(tokens(chunk["text"]))
        scored.append((len(overlap), chunk))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]


def answer(query):
    hits = retrieve(query, k=1)
    score, chunk = hits[0]
    if score == 0:
        return {"answer": "I cannot ground this. Escalate.", "citation": None, "score": 0}
    return {
        "answer": chunk["text"],
        "citation": f"{chunk['source']} ({chunk['id']})",
        "score": score,
    }


print("Four chunks loaded:", [chunk["id"] for chunk in CHUNKS])
print("STOP. Do not run the next cell until we guess.")


## Fill 1 — STOP. Do not scroll.

A vendor asks: **"What is our SOC 2 status?"** Nothing in P1–P4 mentions SOC 2.

Guess to a neighbor (20 seconds): will the agent **invent** an answer, **refuse**, or **escalate**? Two voices share. Then type. Then we run the three questions.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor 20s. Two voices. Then type: "invent", "refuse", or "escalate".
guess_soc2 = ""  # YOU TYPE HERE
print("You said:", guess_soc2 or "(say it, then type it)")
print("Next cell runs Maya, Jordan, and the SOC 2 question. Watch the citation.")


## This cell — three questions. RUN WITH ME


In [ ]:
# RUN WITH ME. Maya's mug, Jordan's change of mind, then a question with no chunk.
for query in [
    "Can we cash-refund a damaged mug after 10 days?",
    "Customer wants cash back because they changed their mind.",
    "What is our vendor's SOC 2 status?",
]:
    result = answer(query)
    print(f"Q: {query}")
    print(f"   score={result['score']} citation={result['citation']}")
    print(f"   {result['answer']}\n")


**Family check.** Say this out loud: this is the **LLM** family. An RL-based agent cannot cite P1. A DAT 5565 embedding can rank chunks — that is a **tool**, not the agent. The agent still has to act (refund, refuse, escalate) and take feedback: was the citation real?

STOP. Next: short-term memory. Then the near miss.


## Short-term memory — still the agent. RUN WITH ME

The policy pages are **long-term memory** (the corpus). After retrieve, the agent must **hold** the citation while it plans the next step. If it drops the citation, the next step invents a window — same as Monday's chatbot inventing $999.

![Hold the citation](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-memory.png)

A Q-table is not this job. Run this cell. Do not edit it.


In [ ]:
# RUN WITH ME. The corpus is long-term. working_memory is what the agent holds.
working_memory = {"citation": None, "plan": ["retrieve", "cite or miss", "recommend", "release"]}

print("Plan (decide):", " → ".join(working_memory["plan"]))
for query in [
    "Can we cash-refund a damaged mug after 10 days?",
    "What is our vendor's SOC 2 status?",
]:
    found = answer(query)
    working_memory["citation"] = found["citation"]
    next_step = "recommend, then release" if found["citation"] else "escalate (empty memory)"
    print(f"Q: {query}")
    print(f"   hold: {working_memory['citation']}")
    print(f"   next: {next_step}\n")
print("STOP. If the sticky is blank, the agent must not send cash.")


**Family check.** Memory here is the citation the LLM-based agent carries. It is not a Q-table. STOP. Do not scroll to the near miss until I say.


## Fill 2 — STOP. A wrong chunk is a management problem.

![Near miss on the word refund](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-near-miss.png)

Question for the retriever: **"Is a refund allowed?"** Both P1 (damage) and P2 (change of mind) share the word refund.

Guess to a neighbor: which chunk comes **first**, P1 or P2? Two voices. Then type. If you speak the wrong sentence to a customer, you have unapproved policy — same cash leak as Monday's $999 chatbot.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor 20s. Two voices. Then type "P1" or "P2".
guess_near = ""  # YOU TYPE HERE
print("You said first chunk:", guess_near or "(P1 or P2, then we run it)")


## This cell — near miss. RUN WITH ME


In [ ]:
# RUN WITH ME. Top two chunks so the class can see a near miss.
print("Near miss for: Is a refund allowed?")
for score, chunk in retrieve("Is a refund allowed?", k=2):
    print(f"  score={score} {chunk['id']}: {chunk['text']}")
print("A citation is not enough. Ask: did that sentence answer THIS question?")


Managers should ask:

- Did the chunk come from the system of record, or from an old PDF?
- Does the reply cite the chunk id?
- What is the cost of the wrong chunk (cash out the door, privacy, a bad promise)?

**Family check.** A Q-table cannot veto a bad citation. Hybrid later: learned policy on the usual case; language agent plus a reviewer on the document.

STOP. Next fill: who owns release?


## Fill 3 — STOP. Who releases the action?

Priya, order 1003: "Should we refund, and who signs?"

| Role | Owns | Must pass on |
| --- | --- | --- |
| Researcher | The cited chunk | citation id or an explicit miss |
| Analyst | The recommendation | action plus the chunk it used |
| Reviewer | The release decision | approve only if the citation is real |

Guess to a neighbor: who **releases** the refund — researcher, analyst, or reviewer? Two voices. Then type.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor 20s. Two voices. Then type: researcher, analyst, or reviewer.
who_releases = ""  # YOU TYPE HERE
print("Release belongs to:", who_releases or "(say it, then type it)")


## This cell — three-role handoff. RUN WITH ME


In [ ]:
# RUN WITH ME. Researcher cites. Analyst recommends. Reviewer owns release.
def researcher(query):
    result = answer(query)
    return {"role": "researcher", "citation": result["citation"], "evidence": result["answer"]}


def analyst(research):
    if research["citation"] is None:
        action = "escalate"
    elif "30 days" in research["evidence"]:
        action = "apply the 30-day damage rule; do not invent a new window"
    else:
        action = "follow the cited sentence only"
    return {"role": "analyst", "action": action, "used": research["citation"]}


def reviewer(research, analysis):
    known_ids = {chunk["id"] for chunk in CHUNKS}
    cited = research["citation"] or ""
    has_real_source = any(chunk_id in cited for chunk_id in known_ids)
    approved = has_real_source and analysis["used"] == research["citation"]
    return {"role": "reviewer", "approved": approved, "reason": "citation matches the corpus" if approved else "missing citation"}


query = "Damaged item refund window"
research = researcher(query)
analysis = analyst(research)
review = reviewer(research, analysis)
print(research)
print(analysis)
print(review)

# Broken handoff: the analyst cites nothing. The reviewer must stop the release.
bad_analysis = {"role": "analyst", "action": "refund everything", "used": None}
print("Broken handoff:", reviewer(research, bad_analysis))


**Family check.** Same as Monday's unknown id 9999: a person (or a reviewer role) owns the exception. The researcher does not get to send cash.

STOP. Next fill is **your** question.


## Fill 4 — STOP. Your question, in your own words.

Guess to a neighbor what you will ask. Two voices. Then type. If the citation is wrong or missing, say so out loud. That is the management problem.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Type a customer question, then run.
my_question = ""  # YOU TYPE HERE
if not my_question.strip():
    print("Type a question, then run this cell again.")
else:
    found = answer(my_question)
    print("Citation:", found["citation"])
    print(found["answer"])
    print("Did the sentence actually answer your question? Say yes or no out loud.")


## Practice — real CFPB credit-card tickets. RUN WITH ME

Monday was a retail returns desk (Lumen Home). Now the **same LLM-based agent** sits on a bank operations desk. State = a real complaint from the [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/) (CC0). Policy chunks are public US rules, not the model's memory.

Do not paste a customer's SSN or an employer file into a public model. These tickets are already public.


In [ ]:
# RUN WITH ME. Real CFPB tickets + public US policy. Same retrieve / answer / reviewer.
from pathlib import Path
import csv
import urllib.request

def course_data(name):
    """Local folder, Canvas data/, or GitHub — so Colab still runs."""
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    url = (
        "https://raw.githubusercontent.com/salihtutun/"
        "Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
    )
    dest_dir = Path("/content") if Path("/content").exists() else Path(".")
    dest = dest_dir / name
    urllib.request.urlretrieve(url, dest)
    return dest


# Public-domain / statutory rules a credit-card ops desk actually cites.
CHUNKS = [
    {"id": "FCBA", "source": "Fair Credit Billing Act", "text": "A cardholder may dispute a billing error in writing within 60 days of the statement. The issuer must investigate. Do not treat this as a change-of-mind refund."},
    {"id": "FCRA", "source": "Fair Credit Reporting Act", "text": "A consumer may dispute incorrect information on a credit report. The furnisher and the bureau must investigate. Do not silently delete a tradeline to make the ticket go away."},
    {"id": "FRAUD", "source": "Identity theft / card opened without consent", "text": "If a card was opened without the consumer's consent, freeze new charges, file an identity-theft report, and escalate to a person. Do not auto-refund from a chatbot."},
    {"id": "PRIV", "source": "Privacy", "text": "Do not paste Social Security numbers, full card numbers, or employer case files into a public chatbot."},
]

tickets = list(csv.DictReader(course_data("cfpb-credit-card-tickets.csv").open(encoding="utf-8")))
print("Loaded", len(tickets), "real CFPB credit-card tickets.")
print("Agent actions on this queue (retrieve, then act):")
for row in tickets[:5]:
    found = answer(row["ticket_text"])
    action = "escalate" if found["citation"] is None else "follow cited rule; reviewer still releases"
    print(f"  {row['complaint_id']} {row['company'][:22]:22} | {row['issue'][:32]}")
    print(f"     citation={found['citation']} -> {action}")
print("STOP. Same loop as Monday: observe with tools, decide, act, take r.")


**Family check.** A credit-card complaint is a **document**. That is the LLM family. An RL agent cannot cite FCBA. Session 4's repeated FX choice is the other family.

## Fill 5 — STOP. Add this to your framework.

![Documents versus a repeated number](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-two-families.png)

Guess to a neighbor: after today, which family owns **documents** in your project — LLM, RL, or hybrid? Two voices. Then type one sentence you will keep.

Hint: documents and exceptions go through retrieval plus a reviewer. A repeated number (overnight FX sleeve, daily order quantity) waits for Session 4. A DAT 5565 model may rank chunks; it is a tool, not the agent.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")


## Walking-out answers — we say these only after the five fills

1. If nothing matches (SOC 2), the **agent** refuses or escalates. That is still action *a*. Do not invent.
2. Overlap on "refund" can rank P1 first. A citation in memory is not enough — did it answer **this** state?
3. The **reviewer** owns release of *a*. The plan is decide, not a new org chart.
4. Retrieve, memory, and plan are how this LLM-based agent observes and decides. Session 4's bandit is the RL family. Both are Agentic AI. Do not mix them because both are "AI."
5. Your framework line: documents and exceptions → this LLM-based agent (retrieve + memory + reviewer). Repeated numeric choice → RL. Hybrid later = learned policy on the usual case, language agent plus a person on the exception.

Fits: high-volume policy lookup with a system of record. Does not fit: a negotiated exception a manager owns, or a repeated inventory quantity.
